# PlantVillageVQA — MiniCPM5-1B + frozen SigLIP, gated cross-attention
Thin Kaggle driver. Add the **PlantVillageVQA** dataset as a Notebook input, enable **GPU T4 x2**.

**Before running:** in *Add-ons → Secrets*, create `GITHUB_TOKEN` (repo read) and `HF_TOKEN`
(HF read), and attach both to this notebook. The first cell clones the project from GitHub
using those secrets, so you don't need to upload the code manually. Then run top-to-bottom.

In [ ]:
# --- secrets + clone the project from GitHub ---
# In Kaggle: Add-ons -> Secrets -> add GITHUB_TOKEN and HF_TOKEN, then attach them to this notebook.
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

import os
REPO = "ryzewtf/GenAI_LAB_CA"
DST = "/kaggle/working/GenAI_LAB_CA"
if not os.path.isdir(DST):
    # token embedded in the URL so a private repo clones non-interactively
    os.system(f"git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {DST}")
%cd {DST}

# make HF_TOKEN available for gated model downloads (MiniCPM) done by from_pretrained
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
!ls

In [ ]:
# --- environment ---
# torch is preinstalled (CUDA build). NO flash-attn.
!pip -q install -U transformers datasets accelerate sentencepiece pillow pyyaml matplotlib huggingface_hub
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
import torch
print('cuda', torch.cuda.is_available(), 'devices', torch.cuda.device_count())
print('capability', torch.cuda.get_device_capability(0), '(7,5) => T4/fp16')

In [ ]:
# --- assets come from an attached Kaggle Dataset (built by upload_to_kaggle.py) ---
# git carries only code; the weights / checkpoint / features / manifests are attached
# here. In the right panel: Add Input -> Datasets -> your 'agrivqa-assets' dataset.
import os
ASSETS = '/kaggle/input/agrivqa-assets'      # <- change if you named the dataset differently
assert os.path.isdir(ASSETS), f'attach the assets dataset; not found at {ASSETS}'
LLM_LOCAL = f'{ASSETS}/models/MiniCPM5-1B'
VIS_LOCAL = f'{ASSETS}/models/siglip'
CKPT      = f'{ASSETS}/runs/exp1/best.pt'
FEATURES  = f'{ASSETS}/cache/siglip_features'
PREPARED  = f'{ASSETS}/prepared'
for p in (LLM_LOCAL, VIS_LOCAL, CKPT, FEATURES, PREPARED):
    print('ok      ' if os.path.exists(p) else 'MISSING ', p)

In [ ]:
# --- point config at the attached assets ---
import yaml, os
cfg_path = 'configs/default.yaml'
cfg = yaml.safe_load(open(cfg_path))

# frozen backbones load from the local mount (no HF download)
cfg['model']['llm_name']    = LLM_LOCAL
cfg['model']['vision_name'] = VIS_LOCAL
# EVAL reads the uploaded features + manifests straight from the (read-only) mount
cfg['cache']['feature_dir'] = FEATURES
cfg['data']['prepared_dir'] = PREPARED
cfg['train']['out_dir']     = '/kaggle/working/runs/exp1'

# The raw PlantVillageVQA dataset is only needed to RE-RUN prepare/cache (cells 5-6)
# or a full training run. For eval it is not required.
DATA_ROOT = '/kaggle/input/datasets/ryzewtf/plantvillagevqa/PlantVillageVQA'
if os.path.isfile(os.path.join(DATA_ROOT, 'PlantVillageVQA.csv')):
    cfg['data']['data_root'] = DATA_ROOT
    cfg['data']['csv_name'] = 'PlantVillageVQA.csv'
    cfg['data']['images_dirname'] = 'Images'
else:
    print('note: raw PlantVillageVQA not attached -> skip cells 5-8; eval below still\n'
          '      works from the uploaded features/manifests.')

# NB: to do a fresh FULL run instead, set these to writable /kaggle/working paths
# (the mount is read-only) BEFORE running prepare/cache/train:
#   cfg['data']['prepared_dir'] = '/kaggle/working/prepared'
#   cfg['cache']['feature_dir'] = '/kaggle/working/cache'

yaml.safe_dump(cfg, open(cfg_path, 'w'))
print('llm      =', cfg['model']['llm_name'])
print('vision   =', cfg['model']['vision_name'])
print('features =', cfg['cache']['feature_dir'])
print('prepared =', cfg['data']['prepared_dir'])

In [ ]:
!python -m data.prepare --config configs/default.yaml --subset-size 30000

In [ ]:
!python -m data.cache_features --config configs/default.yaml

In [ ]:
# sanity: overfit a tiny slice (loss should fall, EM rise)
!python train.py --config configs/default.yaml --overfit 40 --grad-accum 1 --epochs 40

In [ ]:
!python train.py --config configs/default.yaml

In [ ]:
# --- evaluate on both T4s (sharded, batched generation) using the uploaded ckpt ---
!mkdir -p /kaggle/working/runs/exp1
# full model eval (gates on), split across 2 GPUs; metrics saved to JSON
!python eval.py --config configs/default.yaml --ckpt {CKPT} --split test --gpus 2 --out /kaggle/working/runs/exp1/test_metrics.json
# blind-LLM baseline (gates off) -- exposes the language prior
!python eval.py --config configs/default.yaml --ckpt {CKPT} --split test --gpus 2 --blind --out /kaggle/working/runs/exp1/test_metrics_blind.json
# gate-magnitude figure for the model card
!python analyze_gates.py --ckpt {CKPT} --out /kaggle/working/runs/exp1/gates.png